# OpenStroke12 E90/E91 — HVAC vector không layer

Chọn GPU. Chạy ba run `control`, `unified`, `f2` từ cùng E85 để tách cơ chế và loss. Điểm chính: **F2 object toàn trang**, chưa có kết quả v12. Xem docs/OPENSTROKE12.md.

GT được đọc trực tiếp từ labels/val và vectors/val của dataset. Không cần score_pack riêng. Metric được ghi protocol=dataset_vectors.sampled8.v1; chấm một lần mỗi epoch, dùng lại kết quả best khi kết thúc.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
%pip install -q --upgrade --force-reinstall --no-deps "git+https://github.com/Nguyenchitrai62/OpenStroke.git@main"
%pip install -q scipy shapely opencv-python pyyaml matplotlib


In [ ]:
from pathlib import Path
from zipfile import ZipFile
ZIP = Path("/content/drive/MyDrive/Dataset/OpenStroke9/openstroke_hvac7_data.zip")
MODEL = Path("/content/drive/MyDrive/Train/OpenStroke10/e85_graph2/weights/best.pt")
for path in (ZIP, MODEL):
    assert path.is_file(), f"Thiếu file: {path}. Upload/sửa đường dẫn trước khi train."
ROOT = Path("/content/openstroke9_e83")
ROOT.mkdir(exist_ok=True)
with ZipFile(ZIP) as archive:
    archive.extractall(ROOT)
DATA = ROOT / "obb_hvac7/data.yaml"
assert DATA.is_file(), f"Không thấy {DATA}; kiểm tra cấu trúc ZIP."


In [ ]:
import subprocess
RUN = "f2"  # control -> unified -> f2; tất cả khởi tạo cùng MODEL, không resume
options = {"control": (False, False, "e90_control_native"), "unified": (True, False, "e90_unified_native"), "f2": (True, True, "e91_f2_native")}
unified, f2, name = options[RUN]
args = ["openstroke", "obb", "train", f"data={DATA}", f"model={MODEL}",
        "epochs=10", "batch=32", "lr0=0.0001", "warmup_epochs=0.5", "nbs=64", "seed=0",
        "max_iters=190", "stroke_graph=True", "endpoint_members=True",
        "graph_foreground=False", "graph_hard_negative=False", "graph_object_class=False",
        f"graph_unified={unified}", f"graph_f2={f2}", "select_metric=object_f2",
        "imgsz=1024", "tile=True", "tile_scale=1", "max_tokens=8192",
        "project=/content/drive/MyDrive/Train/OpenStroke12", f"name={name}"]
subprocess.run(args, check=True)
